# 🧬 ⚙️ CpGPT Imputation Tutorial ⚙️ 🧬

Welcome to the CpGPT Imputation Tutorial! 👋 

In this notebook, we'll walk you through the process of reconstructing the methylome based on a pre-trained CpGPT model.

## Table of Contents

1. [Setup Environment](#1-setup-environment)
2. [Load CpGPT Dependencies](#2-load-cpgpt-dependencies)
3. [Load Pre-trained Model](#3-load-pre-trained-model)
4. [Save Imputation Data](#4-save-imputation-data)
5. [Create Dataset](#5-create-dataset)
6. [Convert Illumina Probes](#6-convert-illumina-probes)
7. [Generate DNA LLM Embeddings](#7-generate-dna-llm-embeddings)
8. [Embed and Impute Methylome](#8-embed-and-impute-methylome)
9. [Visualize Imputed Methylome](#9-visualize-imputed-methylome)
10. [Next Steps](#10-next-steps)

## 1. Setup Environment

We'll import the necessary Python packages and set up our environment for CpGPT. We'll be using a mix of standard data science libraries and CpGPT-specific modules. We'll also set some important variables that will be used throughout the notebook. Pay attention to these as you may need to adjust them based on your specific setup and requirements. The bulk of the variables are defined here below. The pre-trained model checkpoint alongside DNA embedding files can also be downloaded if not present.

CpGPT model files and DNA-sequence dependencies are hosted on Hugging Face.
The download cell below downloads any missing files into the standard Hugging Face cache and reuses them on later runs; no command-line download is needed.

### 1.1 Download dependencies and define variables

In [ ]:
from cpgpt import download_cpgpt

resources = download_cpgpt(model="hannum", species="human")

In [ ]:
# Random seed for reproducibility
RANDOM_SEED = 42

# Directory paths
PROCESSED_DIR = "../data/tutorials/processed/impute"


ARROW_DF_PATH = "../data/tutorials/raw/toy.arrow"
ARROW_MASKED_DF_PATH = "../data/tutorials/raw/toy_masked.arrow"

# The maximum context length to give to the model
MAX_INPUT_LENGTH = 20_000

# Device configuration
DEVICE = "cuda"

LLM_DEPENDENCIES_DIR = str(resources.dependencies_path)
DEPENDENCIES_DIR = str(resources.dependencies_path)
MODEL_CHECKPOINT_PATH = str(resources.checkpoint_path)
MODEL_CONFIG_PATH = str(resources.config_path)
MODEL_VOCAB_PATH = str(resources.vocab_path) if resources.vocab_path is not None else None

> **⚠️ Warning**
> 
> It is recommended to have a GPU for inference as CPU might be slow.
> 
> Reconstructing the methylome for a few hundred samples might take up to one hour on a CPU. ⌛
>
> This might be a great exercise in testing your patience.

### 1.2 Import packages


In [ ]:
# Standard library imports
import warnings
import os
from pathlib import Path

warnings.simplefilter(action="ignore", category=FutureWarning)

# Plotting imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyaging as pya
import seaborn as sns

# Lightning imports
from lightning.pytorch import seed_everything
from scipy.stats import pearsonr, spearmanr
from sklearn.impute import KNNImputer

# cpgpt-specific imports
from cpgpt.data.components.cpgpt_datasaver import CpGPTDataSaver
from cpgpt.data.components.cpgpt_dataset import CpGPTDataset
from cpgpt.data.components.dna_llm_embedder import DNALLMEmbedder
from cpgpt.data.components.illumina_methylation_prober import IlluminaMethylationProber
from cpgpt.infer.cpgpt_inferencer import CpGPTInferencer
from cpgpt.utils.rich_utils import create_rich_dataset_preview

# Set random seed for reproducibility
seed_everything(RANDOM_SEED, workers=True)

### 1.3 Download toy data


In [16]:
# Only download the dependencies if they don't exist
if not os.path.exists(ARROW_DF_PATH):
    command = f"aws s3 cp s3://cpgpt-lucascamillo-public/data/cpgcorpus/raw/GSE163839/GPL13534/betas/QCDPB.arrow {ARROW_DF_PATH} --request-payer requester"
    os.system(command)

download: s3://cpgpt-lucascamillo-public/data/cpgcorpus/raw/GSE163839/GPL13534/betas/QCDPB.arrow to ../data/tutorials/raw/toy.arrow


## 2. Load CpGPT Dependencies

Now that we have our basic environment set up, we need to load three crucial components for CpGPT:

1. **DNA LLM Embedder**: This component contains the embeddings for all CpGs of interest. It's essential for converting DNA sequences into a format that our model can understand and process.

2. **Illumina Methylation Prober**: This tool allows us to convert probe names to genomic locations, which is required for working with Illumina methylation array data.

3. **CpG Inferencer**: This component is responsible for loading the pre-trained CpGPT model checkpoint and any additional configurations or overrides needed for the fine-tuning process.

### 3.1 Load DNALLMEmbedder

In [21]:
embedder = DNALLMEmbedder(dependencies_dir=DEPENDENCIES_DIR)

cpgpt: DNALLMEmbedder: Initializing class DNALLMEmbedder.
cpgpt: DNALLMEmbedder: Genome files will be stored under ../dependencies/human/genomes.
cpgpt: DNALLMEmbedder: DNA embeddings will be stored under ../dependencies/human/dna_embeddings and subdirectories.
cpgpt: DNALLMEmbedder: Ensembl metadata dictionary loaded successfully


### 3.2 Load IlluminaMethylationProber

In [22]:
prober = IlluminaMethylationProber(dependencies_dir=DEPENDENCIES_DIR, embedder=embedder)

cpgpt: IlluminaMethylationProber: Initializing class IlluminaMethylationProber.
cpgpt: IlluminaMethylationProber: Illumina methylation manifest files will be stored under ../dependencies/human/manifests.
cpgpt: IlluminaMethylationProber: Illumina metadata dictionary loaded successfully.


### 3.3 Load CpGInferencer

In [23]:
inferencer = CpGPTInferencer()

cpgpt: CpGPTInferencer: Initializing class CpGPTInferencer.
cpgpt: CpGPTInferencer: Using device: cuda.


## 3. Load Pre-trained Model

We need to load a pre-trained CpGPT model to be able to reconstruct the beta values for different CpG sites. Generally, the pre-trained model works well for zero-shot imputation but you might want to finetune the model using data that is more appropriate for your imputation goals. For instance, finetuning on RRBS data before following this tutorial.

In [ ]:
config = inferencer.load_cpgpt_config(MODEL_CONFIG_PATH)
model_info = {"current_hparams": config}
model = inferencer.load_cpgpt_model(
    config,
    model_ckpt_path=MODEL_CHECKPOINT_PATH,
    strict_load=True,
)

## 4. Save Imputation Data

We'll prepare our methylation data for reconstruction and save it in a format that CpGPT can efficiently process. We first begin with a dataset that is already stored in .arrow (feather v2) format. Then, let's create a memory-mapped dataset.

For demonstration purposes, let's mask 90% of the data. 

In [ ]:
# Read the arrow file
df = pd.read_feather(ARROW_DF_PATH)

# TODO: Remove this
df = df.iloc[:, -30000:]

# Save age and female columns
age = df["age"]
female = df["female"]

# Filter df to only include CpGs from the Illumina Prober
df = df.loc[
    :,
    np.intersect1d(df.columns, list(prober.illumina_metadata_dict["homo_sapiens"].keys())),
]

# Show first few rows
df.head()

In [ ]:
# Create a random mask for df
mask = np.random.choice([True, False], size=df.shape, p=[0.8, 0.2])

# Apply the mask to create a df with random missing values
df_masked = df.mask(mask)

# Save masked df
df_masked.to_feather(ARROW_MASKED_DF_PATH)

# Show first few rows of the masked subset
df_masked.head()

In [ ]:
# Define datasaver
imputation_datasaver = CpGPTDataSaver(data_paths=ARROW_MASKED_DF_PATH, processed_dir=PROCESSED_DIR)

# Process the file
imputation_datasaver.process_files(prober, embedder)

## 5. Create Dataset

We need to define a way that the CpGPT model can efficiently access our memory-mapped dataset. Let's define a CpGPTDataset object and check the output.

In [ ]:
imputation_data = CpGPTDataset(
    embedder,
    processed_dir=PROCESSED_DIR,
    max_length=MAX_INPUT_LENGTH,
    sorting_strategy=model_info["current_hparams"]["data"]["sorting_strategy"],
    dna_context_len=model_info["current_hparams"]["data"]["dna_context_len"],
    dna_llm=model_info["current_hparams"]["data"]["dna_llm"],
)

Let's double check the outputs make sense:

In [ ]:
create_rich_dataset_preview(imputation_data, "Imputation Data Sample")

## 6. Convert Illumina Probes

In order for the model to understand what part of the methylome needs to be reconstructed, a genomic location is necessary. This can be given directly in the format '12:213103' or first converted from the probes of the Illumina methylation arrays. The IlluminaMethylationProber class has a handy function for that conversion for the most up-to-date arrays as of the publication of this tutorial.

In [ ]:
# Get probes
probes = df.columns

# Convert probes to genomic locations
genomic_locations = prober.locate_probes(probes, "homo_sapiens")

# Show first few locations
genomic_locations[0:5]

## 7. Generate DNA LLM Embeddings
 
In this section, we'll generate any DNA embeddings for the genomic locations of our dataset and for our genomic locations of interest that might be missing. The model requires that all genomic locations have embeddings from a DNA LLM or it will fail otherwise. The easiest way is to download such embeddings if you are working with human samples ([step 1.3](#1.3-download-dependencies)) and Illumina arrays or it might take a little bit for the processing to happen.

In [ ]:
# Embed genomic locations from the imputation data
all_species = list(imputation_datasaver.all_genomic_locations.keys())

# Loop through each species
for species in all_species:
    dataset_genomic_locations = list(imputation_datasaver.all_genomic_locations.get(species, []))

    embedder.parse_dna_embeddings(
        dataset_genomic_locations,
        species,
        dna_llm=model_info["current_hparams"]["data"]["dna_llm"],
        dna_context_len=model_info["current_hparams"]["data"]["dna_context_len"],
    )

## 8. Embed and Impute Methylome

First, let's use CpGPT to create a sample embedding given the available information in the CpG sites. Then, let's ask it to reconstruct the beta values for the genomic locations of interest. We also get the uncertainty for each reconstruction from the model.

### 8.1 Reconstruct the whole methylome

In [ ]:
# Embed samples
sample_embeddings = inferencer.embed_sample(model, imputation_data, batch_size=2)

In [ ]:
# Reconstruct beta values
X_imp, X_unc = inferencer.reconstruct_betas(
    model,
    sample_embedding=sample_embeddings,
    genomic_locations=genomic_locations,
    embedder=embedder,
    dna_llm=model_info["current_hparams"]["data"]["dna_llm"],
    dna_context_len=model_info["current_hparams"]["data"]["dna_context_len"],
    species="homo_sapiens",  # You need to pick a single species
    batch_size=1,
    return_type="numpy",
)

In [ ]:
# Transform to DataFrame
X_imp_df = pd.DataFrame(X_imp, columns=probes, index=df.index)
X_unc_df = pd.DataFrame(X_unc, columns=probes, index=df.index)

### 8.2 Impute with CpGPT, mean and KNN

In [ ]:
# CpGPT Imputation
df_imp_cpgpt = df_masked.copy()
df_imp_cpgpt = df_imp_cpgpt.where(~mask, X_imp_df, axis=1)

# Mean imputation
df_imp_mean = df_masked.copy()
mean_values = df_imp_mean.mean()
df_imp_mean = df_imp_mean.where(~mask, mean_values, axis=1)

# KNN imputation
knn_imputer = KNNImputer(n_neighbors=5)
df_imp_knn = df_masked.copy()
df_imp_knn = pd.DataFrame(
    knn_imputer.fit_transform(df_imp_knn),
    columns=df_imp_knn.columns,
    index=df_imp_knn.index,
)

# Create a dictionary of imputed dataframes
df_imps = {"CpGPT": df_imp_cpgpt, "Mean": df_imp_mean, "KNN": df_imp_knn}

In [ ]:
df_imp_cpgpt.head()

In [ ]:
df_imp_mean.head()

In [ ]:
df_imp_knn.head()

## 9. Visualize Imputed Methylome

As a quick sanity check, let's create some plots to try to understand what is going on. We'll visualize:

1. The bimodal distribution of imputed beta values for each method
2. The distribution of uncertainty with higher uncertainty for values close to 0.5
3. The predictions of epigenetic clocks from the imputed values

These visualizations will help us ensure that our model is working as expected. If not, it might be worth finetuning with the appropriate data.

### 9.1 Distribution of reconstructed beta values

In [ ]:
# Create histograms with density plots for CpGPT, Mean, and KNN imputations

# Set up the matplotlib figure
fig, axs = plt.subplots(1, 3, figsize=(15, 4))

# Select 3 samples randomly
sample_size = 1
sample_indices = np.random.choice(df_masked.index, size=sample_size, replace=False)

for idx, (method, df) in enumerate(df_imps.items()):
    # Filter the dataframe to only include imputed values
    df_imputed = df[mask]

    # Filter the dataframe to only include imputed values for the selected samples
    df_imputed = df_imputed.loc[sample_indices]

    # Melt the dataframe to long format for easier plotting
    df_long = df_imputed.melt(var_name="CpG Site", value_name="Beta Value")

    # Create the histogram and KDE plot
    sns.histplot(
        data=df_long,
        x="Beta Value",
        kde=False,
        stat="density",
        bins=50,
        color="skyblue",
        ax=axs[idx],
    )

    # Calculate mean and median of imputed values
    mean_beta = df_long["Beta Value"].mean()
    median_beta = df_long["Beta Value"].median()

    # Add mean and median lines
    axs[idx].axvline(
        mean_beta,
        color="red",
        linestyle="dashed",
        linewidth=2,
        label=f"Mean: {mean_beta:.3f}",
    )
    axs[idx].axvline(
        median_beta,
        color="green",
        linestyle="dashed",
        linewidth=2,
        label=f"Median: {median_beta:.3f}",
    )

    # Customize the plot
    axs[idx].set_title(f"Distribution of {method} Imputed Beta Values\n({sample_size} Samples)")
    axs[idx].set_xlabel("Beta Value")
    axs[idx].set_ylabel("Density")
    axs[idx].set_xlim(0, 1)
    axs[idx].legend()

# Adjust layout and show the plot
plt.tight_layout()
plt.show()

### 9.2 Uncertainty of reconstructed beta values

In [ ]:
# Create a scatter plot with uncertainty using matplotlib and seaborn, only for masked values

# Filter X_imp_df and X_unc_df to include only masked values
X_imp_masked = X_imp_df[mask]
X_unc_masked = X_unc_df[mask]

# Summarize X_imp_masked and X_unc_masked by feature
X_imp_summary = X_imp_masked.mean().reset_index()
X_imp_summary.columns = ["Feature", "Mean Imputed Value"]

X_unc_summary = X_unc_masked.mean().reset_index()
X_unc_summary.columns = ["Feature", "Mean Uncertainty"]

# Calculate confidence interval using the distribution of X_unc_masked
X_unc_quantiles = X_unc_masked.quantile([0.025, 0.975])
X_unc_lower = X_unc_quantiles.iloc[0].reset_index()
X_unc_upper = X_unc_quantiles.iloc[1].reset_index()
X_unc_lower.columns = ["Feature", "CI_lower"]
X_unc_upper.columns = ["Feature", "CI_upper"]

# Merge the summaries and confidence intervals
merged_summary = pd.merge(X_imp_summary, X_unc_summary, on="Feature")
merged_summary = pd.merge(merged_summary, X_unc_lower, on="Feature")
merged_summary = pd.merge(merged_summary, X_unc_upper, on="Feature")

# Sort by Mean Imputed Value for better visualization
merged_summary = merged_summary.sort_values("Mean Imputed Value")

# Create the scatter plot with uncertainty
plt.figure(figsize=(5, 3))

# Plot the main scatter points
scatter = plt.scatter(
    merged_summary["Mean Imputed Value"],
    merged_summary["Mean Uncertainty"],
    c=merged_summary["Mean Uncertainty"],
    cmap="viridis",
    alpha=0.4,
    s=10,
)

# Add error bars for uncertainty, ensuring non-negative values
yerr_lower = np.maximum(merged_summary["Mean Uncertainty"] - merged_summary["CI_lower"], 0)
yerr_upper = np.maximum(merged_summary["CI_upper"] - merged_summary["Mean Uncertainty"], 0)

plt.errorbar(
    merged_summary["Mean Imputed Value"],
    merged_summary["Mean Uncertainty"],
    yerr=[yerr_lower, yerr_upper],
    fmt="none",
    ecolor="lightgray",
    alpha=0.3,
)

# Customize the plot
plt.title("Uncertainty of Reconstructed Beta Values (Masked Only)")
plt.xlabel("Mean Reconstructed Beta Value")
plt.ylabel("Mean Uncertainty (with 95% CI)")
plt.xlim(0, 1)  # Beta values are between 0 and 1
plt.colorbar(scatter, label="Mean Uncertainty")

# Add grid
plt.grid(True, linestyle="--", alpha=0.7)

# Show the plot
plt.tight_layout()
plt.show()

### 9.3 Epigenetic clock prediction

In [ ]:
# Define clocks of interest from pyaging
clocks = [
    "horvath2013",
    "dnamphenoage",
    "grimage2",
    "epitoc1",
    "hannum",
    "dnamtl",
    "altumage",
    "pchorvath2013",
    "dunedinpace",
    "zhangmortality",
]

# Add age and female columns to the dataframes
df["age"] = age
df["female"] = female
df_imp_cpgpt["age"] = age
df_imp_cpgpt["female"] = female
df_imp_mean["age"] = age
df_imp_mean["female"] = female
df_imp_knn["age"] = age
df_imp_knn["female"] = female

# Predict age using original beta values
adata_original = pya.pp.df_to_adata(df, imputer_strategy="mean", verbose=False)
pya.pred.predict_age(adata_original, clocks, verbose=False)

# Predict age using imputed beta values for each method
adata_imputed_cpgpt = pya.pp.df_to_adata(df_imp_cpgpt, imputer_strategy="mean", verbose=False)
adata_imputed_mean = pya.pp.df_to_adata(df_imp_mean, imputer_strategy="mean", verbose=False)
adata_imputed_knn = pya.pp.df_to_adata(df_imp_knn, imputer_strategy="mean", verbose=False)

pya.pred.predict_age(adata_imputed_cpgpt, clocks, verbose=False)
pya.pred.predict_age(adata_imputed_mean, clocks, verbose=False)
pya.pred.predict_age(adata_imputed_knn, clocks, verbose=False)

# Calculate correlations and MAE for each clock and imputation method
results = {}

# Set up the subplot grid
fig, axs = plt.subplots(len(clocks) // 2, 2, figsize=(15, len(clocks) // 2 * 6))
fig.suptitle("Original vs Imputed Epigenetic Clock Predictions", fontsize=16)

for i, clock in enumerate(clocks):
    if clock in adata_original.obs.columns and clock in adata_imputed_cpgpt.obs.columns:
        original = np.array(adata_original.obs[clock])
        imputed_cpgpt = np.array(adata_imputed_cpgpt.obs[clock])
        imputed_mean = np.array(adata_imputed_mean.obs[clock])
        imputed_knn = np.array(adata_imputed_knn.obs[clock])

        row = i // 2
        col = i % 2

        # Create scatter plot for each imputation method
        axs[row, col].scatter(original, imputed_cpgpt, label="CpGPT", alpha=0.6)
        axs[row, col].scatter(original, imputed_mean, label="Mean", alpha=0.6)
        axs[row, col].scatter(original, imputed_knn, label="KNN", alpha=0.6)

        # Add identity line
        min_val = min(original.min(), imputed_cpgpt.min(), imputed_mean.min(), imputed_knn.min())
        max_val = max(original.max(), imputed_cpgpt.max(), imputed_mean.max(), imputed_knn.max())
        axs[row, col].plot([min_val, max_val], [min_val, max_val], "r--", alpha=0.7)

        axs[row, col].set_xlabel("Original Prediction")
        axs[row, col].set_ylabel("Imputed Prediction")
        axs[row, col].set_title(clock)
        axs[row, col].legend()

        # Calculate and add text annotation with correlations and MAE for each method
        for method, imputed in zip(
            ["CpGPT", "Mean", "KNN"],
            [imputed_cpgpt, imputed_mean, imputed_knn],
            strict=False,
        ):
            pearson_corr, _ = pearsonr(original, imputed)
            spearman_corr, _ = spearmanr(original, imputed)
            mae = np.mean(np.abs(original - imputed))

            if clock not in results:
                results[clock] = {}
            results[clock][method] = {
                "Pearson": pearson_corr,
                "Spearman": spearman_corr,
                "MAE": mae,
            }

            y_pos = (
                0.95 - (len(results[clock]) - 1) * 0.15
            )  # Adjust vertical position for each method
            axs[row, col].text(
                0.05,
                y_pos,
                f"{method}:\nPearson: {pearson_corr:.4f}\nSpearman: {spearman_corr:.4f}\nMAE: {mae:.4f}",
                transform=axs[row, col].transAxes,
                verticalalignment="top",
                bbox={
                    "boxstyle": "round",
                    "facecolor": "white",
                    "edgecolor": "black",
                    "alpha": 0.7,
                },
            )

# Adjust layout
plt.tight_layout(rect=[0, 0.03, 1, 0.95])

plt.show()

## 10. Next steps

Here are some ideas for further exploration:

- Experiment with different input context lengths
- Use various pre-trained CpGPT models and compare their imputation
- Reconstruct genomic regions that are not available in the Illumina methylation arrays